## Parent retrievers

Al fragmentar documentos para su procesamiento y recuperación, a menudo nos enfrentamos a un dilema: 

Por un lado, se podrían preferir documentos más reducidos, de modo que los `embeddings` puedan reflejar su significado de manera más exacta y específica. Cuando un documento es demasiado extenso, existe el riesgo de que los `embeddings` pierdan su significado y precisión.

Por otro lado, es crucial mantener documentos con una longitud considerable para preservar el contexto de cada fragmento, y así garantizar la coherencia e integridad de la información.

`ParentDocumentRetriever` aborda eficazmente esta contradicción al dividir y almacenar fragmentos de datos concisos. Durante el proceso de recuperación, este sistema primero accede a los fragmentos más pequeños y posteriormente identifica y busca los identificadores principales de dichos fragmentos, retornando finalmente los documentos de mayor tamaño. 

Es crucial aclarar que el término "documento principal" hace referencia al documento fuente del que se extrajo un fragmento pequeño. Esto puede ser el documento íntegro original o un segmento más amplio del mismo.

**Ejemplo:**

Por ejemplo, si se está procesando un libro, podríamos querer fragmentar cada capítulo o sección para obtener `embeddings` más precisos sobre los temas tratados en cada uno. En este caso, un capítulo sería un "documento principal", y cada fragmento o sección del capítulo representaría un fragmento más pequeño.

1. **Proceso de Fragmentación:**
   - El libro se divide en capítulos.
   - Cada capítulo se fragmenta en secciones más pequeñas.

2. **Proceso de Recuperación:**
   - `ParentDocumentRetriever` recupera primero las secciones más pequeñas del capítulo.
   - Luego, identifica y recupera el capítulo completo (documento principal) basándose en los fragmentos pequeños.

Este enfoque permite una búsqueda y recuperación de información más eficiente y precisa, asegurando que cada fragmento recuperado mantenga su contexto original y, al mismo tiempo, brinde un entendimiento profundo y detallado de su contenido.


<div style="position:relative;">
  <img src="../docs/media/diagrams/01.png" alt="Parent Retrievers" style="width:800px;"/>
  <span style="font-size:16px; position:absolute; top:155px; left:650px;">User's query</span>
</div>

## Librerías

In [2]:
%%capture
!pip install langchain-text-splitters

In [4]:
from functools import partial

from dotenv import load_dotenv
# from langchain_openai import OpenAIEmbeddings
from langchain_mistralai import MistralAIEmbeddings
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from src.langchain_docs_loader import LangchainDocsLoader, num_tokens_from_string

load_dotenv()

True

## Funciones de utilidad

In [5]:
embeddings = MistralAIEmbeddings(model="mistral-embed")

vector_store = Chroma(
    collection_name="full_documents",
    embedding_function=embeddings,
)

## Carga de datos

In [6]:
loader = LangchainDocsLoader()
docs = loader.load()
len(docs)

120

## Recuperación de los documentos completos

In [8]:
child_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=100,
    chunk_overlap=10,
    length_function=num_tokens_from_string,
)

store = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=store,
    child_splitter=child_splitter,
)

retriever.add_documents(docs[:5])  # Add only first 5 documents for demo purposes

La cantidad de documentos en nuestra `Store` es igual a la cantidad de documentos en nuestro dataset.

In [9]:
len(list(store.yield_keys()))

5

Al buscar documentos directamente en la `VectorStore`, obtendrás fragmentos de documentos que fueron procesados por el `TextSplitter`.

In [10]:
full_documents_similarity = vector_store.similarity_search(
    "Does the MultiQueryRetriever might be able to overcome some of the limitations of...?"
)
full_documents_similarity

[Document(id='6349cc18-c907-4b79-b736-32046e64a93c', metadata={'doc_id': '9fc8d70f-dbfe-402d-81b5-c1173eff14df', 'title': 'Memory overview - Docs by LangChain', 'language': 'en', 'source': 'https://docs.langchain.com/oss/python/concepts/memory'}, page_content='Working with document collections also shifts complexity to memory **search** over the list. The Store currently supports both [semantic search](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.SearchOp.query) and [filtering by content](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.SearchOp.filter).'),
 Document(id='b01e7c79-e91e-47da-98ce-e9345d1ed43a', metadata={'doc_id': '9fc8d70f-dbfe-402d-81b5-c1173eff14df', 'language': 'en', 'title': 'Memory overview - Docs by LangChain', 'source': 'https://docs.langchain.com/oss/python/concepts/memory'}, page_content='However, this method also presents challenges. It may increase complexity if the agent requires a new tool to d

Si ahora realizas una búsqueda en el `ParentDocumentRetriever`, obtendrás los documentos completos.
Esto se debe a que el `ParentDocumentRetriever` primero busca los fragmentos que hacen `match` con la `query`, después busca los documentos completos sin repeticiones y finalmente devuelve el resultado.

In [14]:
full_documents_retriever = retriever.invoke(
    "Does the MultiQueryRetriever might be able to overcome some of the limitations of...?"
)
full_documents_retriever

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/concepts/memory', 'title': 'Memory overview - Docs by LangChain', 'language': 'en'}, page_content='[Memory](/oss/python/langgraph/add-memory) is a system that remembers information about previous interactions. For AI agents, memory is crucial because it lets them remember previous interactions, learn from feedback, and adapt to user preferences. As agents tackle more complex tasks with numerous user interactions, this capability becomes essential for both efficiency and user satisfaction.\n\nThis conceptual guide covers two types of memory, based on their recall scope:\n\n- [Short-term memory](#short-term-memory), or [thread](/oss/python/langgraph/persistence#threads)-scoped memory, tracks the ongoing conversation by maintaining message history within a session. LangGraph manages short-term memory as a part of your agent’s [state](/oss/python/langgraph/graph-api#state). State is persisted to a database using a [checkp

Puedes corroborar que el `ParentDocumentRetriever` está regresando el subconjunto `único` de documentos completos al comparar el número de documentos recuperados por el `VectorStore` y el `ParentDocumentRetriever`.

In [15]:
[doc.metadata["source"] for doc in full_documents_similarity], [
    doc.metadata["source"] for doc in full_documents_retriever
]

(['https://docs.langchain.com/oss/python/concepts/memory',
  'https://docs.langchain.com/oss/python/concepts/memory',
  'https://docs.langchain.com/oss/python/concepts/memory',
  'https://docs.langchain.com/oss/python/concepts/memory'],
 ['https://docs.langchain.com/oss/python/concepts/memory'])

Como se puede apreciar, el número de documentos recuperados por el `ParentDocumentRetriever` es menor al número de documentos recuperados por el `VectorStore`, ya que los documentos completos pueden contener múltiples fragmentos que coinciden con la consulta. En el ejemplo anterior, el `VectorStore` devolvió 4 fragmentos, mientras que el `ParentDocumentRetriever` devolvió solo 1 documento completo.

## Recuperación de fragmentos largos en lugar de documentos completos

Los documentos pueden ser muy grandes para ser recuperados en su totalidad y ser útiles. 

Por ejemplo, un documento completo podría ser un libro, pero quizá sólo necesito un capítulo para responder a mi pregunta. O quizá sólo necesito un par de párrafos.

Si planeas utilizar los documentos recuperados en un proceso de `Retrival Augmented Generation` (RAG), es posible que los documentos gigantes ni siquiera puedan ser procesados por la ventana de contexto del modelo de lenguaje.

Para este caso, el `ParentDocumentRetriever` puede ser configurado para romper los documentos en fragmentos pequeños, buscar sobre ellos y luego devolver fragmentos más largos (sin ser el documento completo).

In [16]:
parent_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=400,
    chunk_overlap=40,
    length_function=num_tokens_from_string,
)

child_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=100,
    chunk_overlap=10,
    length_function=num_tokens_from_string,
)

vectorstore = Chroma(
    collection_name="big_fragments",
    embedding_function=embeddings,
)

store = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

retriever.add_documents(docs[:5])  # Add only first 5 documents for demo purposes

Los documentos padres son 4 veces más largos que los documentos hijos.

Ahora hay más documentos en el `Store` dado que cada documento se ha dividido en fragmentos más pequeños.

In [17]:
len(list(store.yield_keys()))

39

In [18]:
vectorstore.similarity_search(
    "Does the MultiQueryRetriever might be able to overcome some of the limitations of...?"
)

[Document(id='025ba9b1-a27c-42e9-9920-87b0f63566aa', metadata={'language': 'en', 'source': 'https://docs.langchain.com/oss/python/concepts/memory', 'doc_id': '8525f19f-065c-45dc-99f7-aa35d6447eca', 'title': 'Memory overview - Docs by LangChain'}, page_content='Working with document collections also shifts complexity to memory **search** over the list. The Store currently supports both [semantic search](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.SearchOp.query) and [filtering by content](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.SearchOp.filter).'),
 Document(id='60a1f7a3-b190-40f2-9fef-c0c12c28e0f4', metadata={'language': 'en', 'doc_id': 'a26c79da-569e-4366-ad3c-84813f80c4a8', 'source': 'https://docs.langchain.com/oss/python/concepts/memory', 'title': 'Memory overview - Docs by LangChain'}, page_content='However, this method also presents challenges. It may increase complexity if the agent requires a new tool to d

In [19]:
retriever.invoke(
    "Does the MultiQueryRetriever might be able to overcome some of the limitations of...?"
)

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/concepts/memory', 'title': 'Memory overview - Docs by LangChain', 'language': 'en'}, page_content='#### \u200bCollection\n\nAlternatively, memories can be a collection of documents that are continuously updated and extended over time. Each individual memory can be more narrowly scoped and easier to generate, which means that you’re less likely to **lose** information over time. It’s easier for an LLM to generate _new_ objects for new information than reconcile new information with an existing profile. As a result, a document collection tends to lead to [higher recall downstream](https://en.wikipedia.org/wiki/Precision_and_recall).\n\nHowever, this shifts some complexity memory updating. The model must now _delete_ or _update_ existing items in the list, which can be tricky. In addition, some models may default to over-inserting and others may default to over-updating. See the [Trustcall](https://github.com/hinthornw/t

Estos documentos son un subconjunto de los documentos que hubieramos obtenido si hubiéramos buscado directamente en el `VectorStore`. Por lo cual estos documentos ya tienen toda la especificidad y el contexto de nuestro fragmento o documento de tamaño mas grande.